# A synthetic vxpy / suite2p dataset

This notebook builds a small entarchy in the `Suite2PVxPy` schema, fills it with
synthetic two-photon calcium imaging data, and then works through the things you
would actually do with it: queries, DataFrames, parallel analysis, links between
entities, and archiving.

Nothing here touches real recordings, so it runs anywhere in about a minute. The
same code applies unchanged to an entarchy built by `add_animal` / `add_recording`
from real suite2p output.

Needs `entarchy`, `entarchy_vxpy_suite2p`, `numpy`, `pandas` and `matplotlib`.
The archiving section additionally needs `asdf` and is skipped without it.

In [ ]:
import pathlib
import sys

# Runnable straight from a checkout: put the repository root on the path if the
#  package has not been installed with `pip install -e .`
for candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (candidate / 'entarchy_vxpy_suite2p' / 'schema.py').exists():
        sys.path.insert(0, str(candidate))
        break

In [ ]:
import os
import shutil
import tempfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from entarchy.backend import SQLiteBackend
from entarchy_vxpy_suite2p.schema import (Animal, Layer, Phase, Recording, Roi,
                                          Suite2PVxPy)

rng = np.random.default_rng(20260807)

workdir = tempfile.mkdtemp(prefix='entarchy_example_')
path = os.path.join(workdir, 'demo_entarchy')
print(path)

## The schema

`Suite2PVxPy` fixes the entity hierarchy. An `Animal` holds `Recording`s; a
recording holds both the imaging `Layer`s (suite2p planes) and the stimulation
`Phase`s; a layer holds the `Roi`s.

    Animal > Recording > Layer > Roi
                       > Phase

Every entity in that tree takes arbitrary attributes, added whenever you like -
you do not declare them anywhere.

In [ ]:
ent = Suite2PVxPy.create(path, SQLiteBackend(path, dbname='entarchy.db'))
ent

## Synthetic calcium signals

A dF/F trace is Poisson events convolved with an exponential decay, plus noise.
Passing a shared drive to several ROIs makes them a *functional ensemble* whose
traces genuinely correlate, which matters later when we look for correlated
pairs.

In [ ]:
def calcium_trace(n_frames, rate_hz, shared=None, event_rate=0.05, tau=1.6):
    kernel = np.exp(-np.arange(int(tau * rate_hz * 5)) / (tau * rate_hz))
    trace = np.zeros(n_frames)

    n_events = rng.poisson(event_rate * n_frames)
    for onset, amplitude in zip(rng.integers(0, n_frames, n_events),
                                rng.exponential(0.4, n_events) + 0.1):
        end = min(n_frames, onset + len(kernel))
        trace[onset:end] += amplitude * kernel[:end - onset]

    if shared is not None:
        trace += shared * (0.4 + 0.5 * rng.random())

    return (trace + rng.normal(0, 0.025, n_frames)).astype(np.float32)


plt.figure(figsize=(9, 2))
plt.plot(calcium_trace(600, 10.0))
plt.xlabel('frame')
plt.ylabel('dF/F')
plt.title('one synthetic ROI')
plt.tight_layout()

## Building the dataset

Two animals, two recordings each, two imaging planes per recording, and 60 ROIs
per plane. Entities are created inside `with ent:` so the whole block commits as
one transaction.

Attributes go in through the collection rather than one ROI at a time. That
matters: a per-entity write is a round trip each, while a collection write is one
statement per attribute name.

In [ ]:
FRAMES = 600
PHASES = 8
ROIS_PER_LAYER = 60

STIMULI = ['cmn', 'moving_grating', 'looming', 'flash']

for animal_index in range(2):
    rate_hz = float(rng.uniform(8.0, 12.0))

    with ent:
        animal = Animal(ent, _id=f'fish_{animal_index + 1:02d}', _parent=ent.root)
        ent.add_new_entity(animal)
        animal['metadata/strain'] = ['wt', 'tg(elavl3:GCaMP6s)'][animal_index]
        animal['metadata/age_dpf'] = int(rng.integers(5, 9))

    for recording_index in range(2):
        with ent:
            recording = Recording(ent, _id=f'rec_{recording_index + 1:02d}',
                                  _parent=animal)
            ent.add_new_entity(recording)
            recording['imaging_rate'] = rate_hz
            recording['signal_length'] = FRAMES
            recording['ca_times'] = np.arange(FRAMES) / rate_hz

            bounds = np.linspace(0, FRAMES, PHASES + 1).astype(int)
            for phase_index in range(PHASES):
                phase = Phase(ent, _id=f'phase{phase_index}', _parent=recording)
                ent.add_new_entity(phase)
                phase['index'] = phase_index
                phase['ca_start_index'] = int(bounds[phase_index])
                phase['ca_end_index'] = int(bounds[phase_index + 1]) - 1
                phase['display/stimulus'] = STIMULI[phase_index % len(STIMULI)]
                phase['display/azimuth'] = float(rng.choice(np.arange(0, 360, 45)))

        for layer_index in range(2):
            with ent:
                layer = Layer(ent, _id=f'plane{layer_index}', _parent=recording)
                ent.add_new_entity(layer)
                layer['depth'] = float(layer_index * 15)
                layer['roi_num'] = ROIS_PER_LAYER

                rois = [Roi(ent, _id=f'Roi_{i}', _parent=layer)
                        for i in range(ROIS_PER_LAYER)]
                for roi in rois:
                    ent.add_new_entity(roi)

            # One shared drive per layer, given to a third of its ROIs
            drive = calcium_trace(FRAMES, rate_hz, event_rate=0.08)
            in_ensemble = rng.random(ROIS_PER_LAYER) < 0.33

            records = {
                'index': list(range(ROIS_PER_LAYER)),
                'dff': [calcium_trace(FRAMES, rate_hz,
                                      shared=drive if member else None)
                        for member in in_ensemble],
                'quality': [str(q) for q in
                            rng.choice(['poor', 'fair', 'good'], ROIS_PER_LAYER,
                                       p=[0.2, 0.45, 0.35])],
                'has_receptive_field': [bool(v) for v in
                                        rng.random(ROIS_PER_LAYER) < 0.35],
                's2p/npix': [int(v) for v in rng.integers(25, 180, ROIS_PER_LAYER)],
                'ants/z': [float(layer_index * 15 + v)
                           for v in rng.normal(0, 2, ROIS_PER_LAYER)],
            }

            collection = ent.get(Roi, f'[Layer]uuid == "{layer.uuid}"')
            collection.update(pd.DataFrame(records,
                                           index=[roi.uuid for roi in rois]))

print('done')

In [ ]:
for entity_type in (Animal, Recording, Phase, Layer, Roi):
    print(f'{entity_type.__name__:<10} {len(ent.get(entity_type)):>5}')

## Moving around the tree

Entities know their parent and their path. The schema also adds convenience
properties, so a recording can hand you its ROIs directly.

In [ ]:
roi = ent.get(Roi)[0]

print('id     ', roi.id)
print('parent ', roi.parent.id, '->', roi.parent.parent.id, '->', roi.parent.parent.parent.id)
print('path   ', roi.path)
print('layer  ', roi.layer.id, 'at depth', roi.layer['depth'])
print('animal ', roi.animal.id, roi.animal['metadata/strain'])

In [ ]:
recording = ent.get(Recording)[0]

print('rois in this recording  ', len(recording.rois))
print('phases in this recording', len(recording.phases))
print('layers                  ', len(recording.layers))

## Queries

`ent.get(EntityType, expression)` returns a `Collection`. Expressions filter on
attributes, and combine with `AND`, `OR`, `XOR`, `NOT`, `IN` and `EXIST`.

In [ ]:
print(len(ent.get(Roi, 'has_receptive_field == True')))
print(len(ent.get(Roi, 'quality == "good" AND s2p/npix > 100')))
print(len(ent.get(Roi, 'index IN (0, 1, 2)')))
print(len(ent.get(Roi, 'NOT(EXIST(never_written))')))

Attributes of an ancestor are addressed by type in brackets, or with `../` per
level up. These reach through the hierarchy in one query rather than looping in
Python.

In [ ]:
print(len(ent.get(Roi, '[Animal]metadata/strain == "wt"')))
print(len(ent.get(Roi, '../depth > 0')))
print(len(ent.get(Roi, '[Recording]imaging_rate > 9.0 AND has_receptive_field == True')))

Collections support set operations, so a query can be built up in pieces.

In [ ]:
responsive = ent.get(Roi, 'has_receptive_field == True')
good = ent.get(Roi, 'quality == "good"')

print('responsive        ', len(responsive))
print('good              ', len(good))
print('both              ', len(responsive & good))
print('either            ', len(responsive | good))
print('responsive not good', len(responsive & ~good))

## DataFrames

`dataframe_of` pulls named attributes for a whole collection in one query,
indexed by uuid. Ancestor attributes may be requested as columns too.

In [ ]:
frame = responsive.dataframe_of(['index', 'quality', 's2p/npix', 'ants/z',
                                '[Animal]metadata/strain'])
frame.head()

In [ ]:
frame.groupby('[Animal]metadata/strain')['s2p/npix'].describe()

## Reading arrays

Array attributes are stored as blobs and come back as ordinary numpy arrays.
Only the ones you ask for are read.

In [ ]:
sample = responsive[0:6]
traces = np.stack([roi['dff'] for roi in sample])

print(traces.shape, traces.dtype)

fig, axes = plt.subplots(len(sample), 1, figsize=(9, 6), sharex=True, sharey=True)
for axis, trace, roi in zip(axes, traces, sample):
    axis.plot(trace, linewidth=0.8)
    axis.set_ylabel(roi.id, rotation=0, ha='right', va='center')
axes[-1].set_xlabel('frame')
fig.tight_layout()

## Analysis in parallel

`map_async` applies a function to every entity of a collection across worker
processes, committing as it goes.

Workers are started with `spawn`, so they must be able to import the function. A
function defined in a notebook cell lives in a `__main__` they cannot import;
entarchy detects that and ships the definition by value with `cloudpickle`, or
falls back to running in this process with a warning. For production pipelines
keep analysis functions in a module - the real ones live in
`entarchy_vxpy_suite2p.analysis.cmn.functions`.

In [ ]:
def summarize_activity(roi):
    dff = np.asarray(roi['dff'])

    roi['dff_std'] = float(dff.std())
    roi['dff_peak'] = float(dff.max())
    roi['active_fraction'] = float((dff > 3 * dff.std()).mean())


ent.get(Roi).map_async(summarize_activity, _worker_num=2, _calibrate=False)

In [ ]:
ent.get(Roi).dataframe_of(['dff_std', 'dff_peak', 'active_fraction']).describe()

In [ ]:
busy = ent.get(Roi, 'dff_peak > 1.0 AND quality == "good"')
print(len(busy), 'bright, well segmented ROIs')

## Links

A link connects two entities and carries data about the *pair* - something that
belongs to neither end alone. It is what the parent hierarchy cannot express: a
ROI's response to a stimulation phase, or a correlation between two ROIs.

The kind of link is data rather than a class, so you invent one whenever you need
it. What it may connect is recorded and checked on every write.

In [ ]:
ent.define_link_type('mean_response', Phase, Roi,
                     description='trial-averaged dF/F during the phase')
ent.define_link_type('correlated', Roi, Roi, symmetric=True,
                     description='Pearson r of dF/F, kept where |r| > 0.6')

for spec in ent.link_types():
    print(f'{spec.name:<15} {spec.linker} -> {spec.linked}'
          f'{"  (symmetric)" if spec.symmetric else ""}')

`symmetric` only has to be stated when both ends are the same type. Otherwise
the endpoint types already say which end is which, and arguments given the wrong
way round are oriented rather than rejected.

Links are created in bulk from a DataFrame with `linker_uuid` and `linked_uuid`
columns; every other column becomes an attribute of the link.

In [ ]:
recording = ent.get(Recording)[0]
phases = list(recording.phases)
rois = list(recording.rois)[:30]

rows = []
for phase in phases:
    start, end = phase['ca_start_index'], phase['ca_end_index']
    for roi in rois:
        window = np.asarray(roi['dff'])[start:end + 1]
        rows.append({'linker_uuid': phase.uuid, 'linked_uuid': roi.uuid,
                     'mean_dff': float(window.mean()),
                     'peak_dff': float(window.max())})

result = ent.link_from_frame(pd.DataFrame(rows), 'mean_response')
print(result)

### Querying links

`ent.links(kind, expression)` returns a `LinkCollection`. A bare name is an
attribute of the link itself; `@` addresses one of its endpoints, either by
entity type or by role (`@linker`, `@linked`, `@either`, `@both`).

In [ ]:
print(len(ent.links('mean_response')))
print(len(ent.links('mean_response', 'mean_dff > 0.1')))
print(len(ent.links('mean_response', '@Phase.display/stimulus == "looming"')))
print(len(ent.links('mean_response',
                    '@Roi.has_receptive_field == True AND peak_dff > 0.5')))

Endpoint filters reach through the hierarchy as well, so a link can be filtered
by something several levels above one of its ends.

In [ ]:
print(len(ent.links('mean_response', '@linker.[Animal]metadata/strain == "wt"')))
print(len(ent.links('mean_response', '@Roi.../depth == 0.0')))

A `LinkCollection` behaves like any other collection, so the result chains.

In [ ]:
strong = ent.links('mean_response', '@Phase.display/stimulus == "looming"')
strong.where('mean_dff > 0.05').dataframe_of(['mean_dff', 'peak_dff']).describe()

### A collection against itself

For a pairwise quantity, build the matrix and hand it over with a threshold. The
predicate is required: storing every pair is what turns a pairwise result into
millions of rows, and links cost roughly 1.5 kB each against 4 bytes for a
`float32` in a matrix.

In [ ]:
layer = ent.get(Layer)[0]
layer_rois = list(layer.rois)
traces = np.stack([np.asarray(roi['dff']) for roi in layer_rois])

result = ent.link_from_matrix(layer_rois, layer_rois, np.corrcoef(traces),
                              'correlated',
                              where=lambda v: np.abs(v) > 0.6, value_name='r')
print(result)

Given a subset of ROIs, `links(within=True)` asks for the links *among* them -
the collection against itself - while `links()` asks for every link touching it.
Membership is a subquery, so it inherits whatever filter the collection carries.

In [ ]:
subset = ent.get(Roi, 'quality == "good"')

print('touching the subset', len(subset.links('correlated')))
print('among the subset   ', len(subset.links('correlated', within=True)))
print('and strong         ', len(subset.links('correlated', 'r > 0.8', within=True)))

In [ ]:
pairs = subset.links('correlated', within=True)

for link in pairs[0:5]:
    print(f'{link.linker.id:>8} <-> {link.linked.id:<8}  r = {link["r"]:.2f}')

Links also work from a single entity, in either direction.

In [ ]:
roi = layer_rois[0]

print('correlated partners', len(roi.links('correlated')))
print('kinds on this roi  ', roi.link_types())

## Archiving

`to_asdf` writes an entarchy, or a filtered collection, to a self-describing
[ASDF](https://asdf-standard.readthedocs.io) archive. The archive *is* an
entarchy directory, so it opens with the same code - analysis and figure scripts
need no changes to read one.

In [ ]:
from entarchy.backend import asdf_store

archive_path = os.path.join(workdir, 'demo_archive')

if asdf_store.available():
    stats = ent.get(Roi, 'has_receptive_field == True').to_asdf(archive_path)
else:
    print('asdf is not installed - pip install entarchy[asdf]')

In [ ]:
if asdf_store.available():
    archived = Suite2PVxPy(archive_path)

    print('rois in archive  ', len(archived.get(Roi)))
    print('same query works ', len(archived.get(Roi, '[Animal]metadata/strain == "wt"')))
    print('arrays intact    ', archived.get(Roi)[0]['dff'].shape)

    archived.backend.close()
    asdf_store.close_asdf_files()

Archives are read-only. Exporting a collection brings its ancestors along, so
parent lookups and `[Animal]...` filters still resolve inside the archive.

## Cleanup

In [ ]:
from entarchy.core.entity import shutdown_worker_pool

shutdown_worker_pool()
ent.backend.close()
shutil.rmtree(workdir, ignore_errors=True)
print('removed', workdir)